# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../../05_src/.env
%dotenv ../05_src/.secrets

cannot find .env file


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "./documents/ai_report_2025.pdf"
pdf_loader=PyPDFLoader(file_path)
pdf_doc=pdf_loader.load()

document_text= ""
for page in pdf_doc:
    document_text+= page.page_content + "/n"

In [3]:
document_text[:500]

'pg. 1 \n \n \nThe GenAI Divide  \nSTATE OF AI IN \nBUSINESS 2025 \n \n \n \n \n \n \nMIT NANDA \nAditya Challapally \nChris Pease \nRamesh Raskar \nPradyumna Chari \nJuly 2025/npg. 2 \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nNOTES \nPreliminary Findings from AI Implementation Research from Project NANDA \nReviewers: Pradyumna Chari, Project NANDA \nResearch Period: January – June 2025 \nMethodology: This report is based on a multi-method research design that includes \na systematic review of over 300 publicly disclosed AI i'

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [4]:
import os
os.getenv("API_GATEWAY_KEY")

from openai import OpenAI
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

In [5]:
from IPython.display import display, Markdown
system_prompt = "You are a helpful assistant that extracts information from a report and provides a concise summary for an AI professional's development."
prompt = f"""
    Given the following context from a report, do the following:
    
    1. Find the names report's authors.
    2. Find the report's title.
    3. Identify the relevance in a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    4. Generate a summary that is a concise and succinct no longer than 1000 tokens.
    5. Use formal academic writing style.


        
    The report is the following: 
    <report>
    {document_text}
    </report>

    Provide your response in the following format:
    Author: <author>
    Title: <title>
    Relevance: <relevance statement>
    Summary: <summary>
    Tone: <tone>
"""

# Using the system prompt to provide context to the model about its role and the task at hand, which can help improve the quality of the response.
response = client.responses.create(
    model="gpt-4o",
    instructions = system_prompt,
    input = prompt,
)

In [6]:
display(Markdown(response.output[0].content[0].text))
display(Markdown(f"### Input Tokens: {response.usage.input_tokens}"))
display(Markdown(f"### Output Tokens: {response.usage.output_tokens}"))

Author: Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari

Title: The GenAI Divide: State of AI in Business 2025

Relevance: This report is highly relevant for an AI professional because it addresses the critical challenge of the GenAI Divide, highlighting the vast gap between AI adoption and meaningful business transformation. The insights into successful strategies, such as building systems that learn and adapt, guide AI professionals in effectively implementing AI technologies that deliver real business value and staying ahead in a rapidly evolving AI landscape.

Summary: The report, "The GenAI Divide: State of AI in Business 2025," authored by Aditya Challapally, Chris Pease, Ramesh Raskar, and Pradyumna Chari, identifies a significant disparity between AI investment and transformative impact across enterprises. Despite substantial investments, only 5% of AI pilots deliver measurable value, a phenomenon termed the "GenAI Divide." The report reveals that this divide is not due to model quality or regulations but a lack of systems capable of learning and integrating effectively. Key findings highlight user preference for standard tools like ChatGPT due to their flexibility, while custom enterprise solutions struggle with adaptability. Moreover, successful AI integration aligns with business needs, employs external partnerships, and focuses on practical workflow integration. Winners in this AI race prioritize systems that learn and adapt, fulfill niche but impactful use cases, and leverage trust and existing vendor relationships. The report emphasizes the urgency for enterprises to adopt adaptive AI systems before the market solidifies around early adopters and innovators. Future prospects involve the development of an "Agentic Web," where interconnected AI agents autonomously navigate complex digital ecosystems, promising transformative shifts in business processes and enterprise structures.

Tone: Formal academic

### Input Tokens: 10900

### Output Tokens: 356

In [7]:
import os
from pydantic import BaseModel
from openai import OpenAI

# -----------------------------
# Environment / Client Setup
# -----------------------------
client = OpenAI(
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    api_key='any value',
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")}
)

# -----------------------------
# Pydantic Model
# -----------------------------
class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str


# -----------------------------
# Inputs
# -----------------------------
#document_text = document_text

# -----------------------------
# Prompts
# -----------------------------
developer_instructions = """
You are an AI assistant that extracts structured information from text.

Requirements:
- Identify the author(s) of the document
- Identify the title of the document
- Write a relevance statement (one paragraph maximum) explaining why this article is relevant for an AI professional's development
- The Relevance must:
    * Be no longer than one paragraph
    * Be written in "Formal Academic Writing" style
- Return ONLY valid JSON with keys:
    Author, Title, Relevance
"""

user_prompt = f"""
Analyze the following document and extract the required structured output:

DOCUMENT:
{document_text}
"""

# -----------------------------
# API Call
# -----------------------------
response = client.chat.completions.create(
    model="gpt-4o-mini",  # NOT GPT-5 family
    messages=[
        {"role": "developer", "content": developer_instructions},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.3
)

# -----------------------------
# Parse Output into Pydantic
# -----------------------------
import json

raw_output = response.choices[0].message.content
parsed_json = json.loads(raw_output)

article_summary = ArticleSummary(**parsed_json)

# -----------------------------
# Result
# -----------------------------
print(article_summary)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
